# Práctica guiada — Inferencia y contratos

Este notebook es para el alumnado. Completa las celdas marcadas con `TODO` y responde las preguntas antes de mirar la solución docente.

**Objetivo:** pasar de una fila CSV a una especificación clara para el módulo de inferencia de la clase 2.

## Cómo trabajar

1. Copia esta carpeta como tu espacio de trabajo.
2. Ejecuta `uv sync` desde `problem/starter/` cuando empiece la clase 2.
3. En esta práctica no implementes todavía el CLI completo.
4. Trabaja en parejas y cambia quién escribe cada diez minutos.
5. No abras `solutions/` hasta el debrief.

In [16]:
from dataclasses import dataclass
import csv
from pathlib import Path
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field, ValidationError

DATA_PATH = Path("../../../../../../assets/03-wine-quality/inference_samples.csv")
print(DATA_PATH)
print(DATA_PATH.exists())

..\..\..\..\..\..\assets\03-wine-quality\inference_samples.csv
True


## 1. Leer los datos nuevos

### Práctica A — Inspección del CSV

Completa la celda siguiente para:

- Leer el CSV con `csv.DictReader`.
- Imprimir el número de filas.
- Imprimir las columnas.
- Guardar la primera fila en una variable llamada `row`.

In [17]:
# TODO: implementa la lectura del CSV.
# Pista: usa DATA_PATH.open(newline="", encoding="utf-8")
rows = []
row = {}

with DATA_PATH.open(newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        rows.append(row)

print(f"Primera fila: {rows[0]}")
print(f"Número de filas: {len(rows)}")
print(f"Número de columnas: {len(rows[0].keys())}")



Primera fila: {'sample_id': 'red-001', 'fixed_acidity': '7.4', 'volatile_acidity': '0.7', 'citric_acid': '0.0', 'residual_sugar': '1.9', 'chlorides': '0.076', 'free_sulfur_dioxide': '11.0', 'total_sulfur_dioxide': '34.0', 'density': '0.9978', 'ph': '3.51', 'sulphates': '0.56', 'alcohol': '9.4'}
Número de filas: 5
Número de columnas: 12


### Preguntas

1. ¿Qué campo identifica la muestra pero no es una característica del modelo?

El campo que identifica la muestra pero no es una característica del modelo es el `sample_id`. Este campo sirve como identificador para cada fila de datos, pero no proporciona información relevante para el análisis.

2. ¿Cuántas características numéricas hay?

Hay 11, todas las demás columnas, excepto `sample_id`.

3. ¿Qué columna extra añadirías para probar el contrato?

Una columna extra que podrías añadir para probar el contrato sería una columna de etiquetas, que indique a qué grupo o categoría pertenece cada muestra.

4. ¿Qué valor de `ph` sería claramente inválido?

Un valor de `ph` inferior a 0 sería claramente invalido, ya que no puede ser negativo.

## 2. Construir el contrato con Pydantic

El contrato del modelo contiene estas características:

| Campo | Rango orientativo |
|---|---|
| `fixed_acidity` | 0–20 |
| `volatile_acidity` | 0–2 |
| `citric_acid` | 0–2 |
| `residual_sugar` | 0–20 |
| `chlorides` | 0–1 |
| `free_sulfur_dioxide` | 0–100 |
| `total_sulfur_dioxide` | 0–300 |
| `density` | 0.98–1.01 |
| `ph` | 2.5–4.5 |
| `sulphates` | 0–3 |
| `alcohol` | 5–20 |

## Tabla contrato de entrada


| Variable | Tipo | Unidad | Rango razonable | ¿Admite nulos? | Tratamiento si falta |
|---|---|---|---|---|---|
| `fixed_acidity` | `float` | g/L | 0–20 | No | Rechazar la fila |
| `volatile_acidity` | `float` | g/L | 0–2 | No | Rechazar la fila |
| `citric_acid` | `float` | g/L | 0–2 | No | Rechazar la fila |
| `residual_sugar` | `float` | g/L | 0–20 | No | Rechazar la fila |
| `chlorides` | `float` | g/L | 0–1 | No | Rechazar la fila |
| `free_sulfur_dioxide` | `float` | mg/L | 0–100 | No | Rechazar la fila |
| `total_sulfur_dioxide` | `float` | mg/L | 0–300 | No | Rechazar la fila |
| `density` | `float` | g/cm³ | 0.98–1.01 | No | Rechazar la fila |
| `ph` | `float` | escala de pH | 2.5–4.5 | No | Rechazar la fila |
| `sulphates` | `float` | g/L | 0–3 | No | Rechazar la fila |
| `alcohol` | `float` | % vol | 5–20 | No | Rechazar la fila |

`sample_id` identifica la muestra, pero no se utiliza como característica del
modelo.

In [18]:
class WineQualityRequestPydantic(BaseModel):
    # TODO: prohíbe campos extra con model_config.
    # TODO: declara las 11 características con Field(ge=..., le=...).
    model_config = ConfigDict(extra="forbid")

    fixed_acidity: float = Field(ge=0, le=20)
    volatile_acidity: float = Field(ge=0, le=2)
    citric_acid: float = Field(ge=0, le=2)
    residual_sugar: float = Field(ge=0, le=20)
    chlorides: float = Field(ge=0, le=1)
    free_sulfur_dioxide: float = Field(ge=0, le=100)
    total_sulfur_dioxide: float = Field(ge=0, le=300)
    density: float = Field(ge=0.98, le=1.01)
    ph: float = Field(ge=2.5, le=4.5)
    sulphates: float = Field(ge=0, le=3)
    alcohol: float = Field(ge=5, le=20)

# TODO: separa sample_id de la fila y valida solo las características del modelo.
# model_row = ...
# request = WineQualityRequestPydantic.model_validate(model_row)

model_rows = []
for i in rows:
    i.pop("sample_id", None)
    request = WineQualityRequestPydantic.model_validate(i)
    model_rows.append(request)
print(model_rows)

[WineQualityRequestPydantic(fixed_acidity=7.4, volatile_acidity=0.7, citric_acid=0.0, residual_sugar=1.9, chlorides=0.076, free_sulfur_dioxide=11.0, total_sulfur_dioxide=34.0, density=0.9978, ph=3.51, sulphates=0.56, alcohol=9.4), WineQualityRequestPydantic(fixed_acidity=7.8, volatile_acidity=0.88, citric_acid=0.0, residual_sugar=2.6, chlorides=0.098, free_sulfur_dioxide=25.0, total_sulfur_dioxide=67.0, density=0.9968, ph=3.2, sulphates=0.68, alcohol=9.8), WineQualityRequestPydantic(fixed_acidity=11.2, volatile_acidity=0.28, citric_acid=0.56, residual_sugar=1.9, chlorides=0.075, free_sulfur_dioxide=17.0, total_sulfur_dioxide=60.0, density=0.998, ph=3.16, sulphates=0.58, alcohol=9.8), WineQualityRequestPydantic(fixed_acidity=7.3, volatile_acidity=0.65, citric_acid=0.0, residual_sugar=1.2, chlorides=0.065, free_sulfur_dioxide=15.0, total_sulfur_dioxide=21.0, density=0.9946, ph=3.39, sulphates=0.47, alcohol=10.0), WineQualityRequestPydantic(fixed_acidity=7.8, volatile_acidity=0.58, citric

### Práctica B — Validación

Completa una prueba para cada caso:

- Una fila válida.
- Una columna `unexpected_field`.
- Un `ph` fuera de rango.

Antes de ejecutar, escribe qué error esperas.

In [19]:
# TODO: completa los casos y captura ValidationError.
# fila valida, si está bien aparece el mensaje "fila validada: ..."
try:
    request = WineQualityRequestPydantic.model_validate(rows[0])
    print(f"fila validada: {request}")
except ValidationError as error:
    print(error)

# columna inválida, si está mal aparece el mensaje "columna inválida: ..."
invalid_row = rows[0]
invalid_row["sample_id"] = "red-001"
try:
    request = WineQualityRequestPydantic.model_validate(invalid_row)
except ValidationError as error:
    print(f"columna inválida: {error}")

rows[0].pop("sample_id", None)

# ph fuera de rango, si está mal aparece el mensaje "ph fuera de rango: ..."
invalid_row = rows[0]
invalid_row["ph"] = -1.0
try:
    request = WineQualityRequestPydantic.model_validate(invalid_row)
except ValidationError as error:
    print(f"ph fuera de rango: {error}")




fila validada: fixed_acidity=7.4 volatile_acidity=0.7 citric_acid=0.0 residual_sugar=1.9 chlorides=0.076 free_sulfur_dioxide=11.0 total_sulfur_dioxide=34.0 density=0.9978 ph=3.51 sulphates=0.56 alcohol=9.4
columna inválida: 1 validation error for WineQualityRequestPydantic
sample_id
  Extra inputs are not permitted [type=extra_forbidden, input_value='red-001', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden
ph fuera de rango: 1 validation error for WineQualityRequestPydantic
ph
  Input should be greater than or equal to 2.5 [type=greater_than_equal, input_value=-1.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than_equal


## 3. Comparar con `dataclass`

Construye la misma estructura con `dataclass`. No añadas todavía validación manual: observa qué garantiza Python y qué no garantiza.

In [20]:
@dataclass(frozen=True, slots=True)
class WineQualityRequestDataclass:
    # TODO: declara las mismas 11 características.
    fixed_acidity: float
    volatile_acidity: float
    citric_acid: float
    residual_sugar: float
    chlorides: float
    free_sulfur_dioxide: float
    total_sulfur_dioxide: float
    density: float
    ph: float
    sulphates: float
    alcohol: float

# TODO: crea una instancia válida y prueba qué ocurre con ph=99.0.

row_invalid = rows[0].copy()
row_invalid["ph"] = 99.0

# Crear instancia
wine_invalid = WineQualityRequestDataclass(**row_invalid)
print(wine_invalid)

#  Como no se ha establecido un rango de valores, ph puede ser cualquier valor, por lo que no se lanzará ninguna excepción

WineQualityRequestDataclass(fixed_acidity='7.4', volatile_acidity='0.7', citric_acid='0.0', residual_sugar='1.9', chlorides='0.076', free_sulfur_dioxide='11.0', total_sulfur_dioxide='34.0', density='0.9978', ph=99.0, sulphates='0.56', alcohol='9.4')


### Decisión de diseño

Completa la frase:

> Usaría Pydantic en la validación de datos porque permite definir tipos de datos y validarlos automáticamente tanto para entrada como para salida de datos. Usaría `dataclass` en la representación de datos sencillos porque permite definir estructuras de datos simples y eficientes pero no permite la validación automática.

## 4. Definir el orden del vector

El modelo no recibe un diccionario. Recibe una lista numérica con un orden estable.

In [21]:
# TODO: rellena el orden exacto a partir del CSV y del contrato.
FEATURE_NAMES = (
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol",
)

# TODO: construye vector usando getattr(request, name).
vectors = []

for row in rows:
    vector = []

    for name in FEATURE_NAMES:
        vector.append(float(row[name]))

    vectors.append(vector)

print(vectors)

[[7.4, 0.7, 0.0, 1.9, 0.076, 11.0, 34.0, 0.9978, -1.0, 0.56, 9.4], [7.8, 0.88, 0.0, 2.6, 0.098, 25.0, 67.0, 0.9968, 3.2, 0.68, 9.8], [11.2, 0.28, 0.56, 1.9, 0.075, 17.0, 60.0, 0.998, 3.16, 0.58, 9.8], [7.3, 0.65, 0.0, 1.2, 0.065, 15.0, 21.0, 0.9946, 3.39, 0.47, 10.0], [7.8, 0.58, 0.02, 2.0, 0.073, 9.0, 18.0, 0.9968, 3.36, 0.57, 9.5]]


### Práctica C — Error de orden

Intercambia mentalmente `density` y `alcohol`. Responde:

- ¿El vector sigue teniendo 11 valores?

Sí. Solo se han intercambiado dos valores de posición.

- ¿El clasificador detectaría necesariamente el error?

Depende de los valores que se introduzcan en cada posición. Si por ejemplo para la columna density, se le introduce un valor (que sería de alcohol) de 12, el clasificador podría generar una predicción incorrecta, pero no necesariamente detectaría que hay un error.

- ¿Cómo evitarías que cada script definiera un orden distinto?

Para evitar que cada script defina un orden distinto, se podría definir un FEATURE_NAMES y construir el vector a partir de esa lista. Esto garantizaría que todos los modelos sigan el mismo orden, independientemente del script utilizado para preprocesar los datos.

## 5. Contrato de salida

Diseña la estructura que devolverá el módulo. Debe permitir responder:

- ¿Qué muestra se ha procesado?
- ¿Qué categoría se ha predicho?
- ¿Con qué confianza?
- ¿Qué versión de modelo y preprocesado se utilizaron?

In [22]:
class WineQualityPredictionPydantic(BaseModel):
    # TODO: declara los campos de salida y sus restricciones.
    model_config = ConfigDict(extra="forbid")

    sample_id: str = Field(min_length=1)
    quality_band: Literal["needs_review", "acceptable", "excellent"]
    confidence: float = Field(ge=0, le=1)
    model_version: str = Field(min_length=1)
    preprocessing_version: str = Field(min_length=1)
    

# TODO: escribe un ejemplo de salida para red-001.


prediction = WineQualityPredictionPydantic(
    sample_id="red-001",
    quality_band="needs_review",
    confidence=0.9,
    model_version="v1.0",
    preprocessing_version="v1.0",
    
)

print(prediction.model_dump())

{'sample_id': 'red-001', 'quality_band': 'needs_review', 'confidence': 0.9, 'model_version': 'v1.0', 'preprocessing_version': 'v1.0'}


## 6. Demo del docente: observar la inferencia

El docente ejecutará el notebook resuelto y el comando de referencia. Mientras tanto, anota el recorrido:

```text
CSV → contrato → preprocesado → modelo cargado → predicción
```

No copies la implementación. Tu código de producción empieza en la clase 2 con el starter.

En primer lugar se deberá obtener un csv con diferentes columnas para poder llevar a cabo el análisis e inferencia. 

Posteriormente, en el script del contrato, se deberá declarar la estructura de los datos que se espera recibir mediante pydantic. Esto incluye definir los tipos de datos y los rangos válidos para cada variable.

En el script de preprocesado, se deberá implementar la lógica para transformar los datos de entrada en el formato esperado por el modelo, transformando las variables de entrada en un vector cuyo orden será el de las variables definidas en FEATURE_NAMES. 

En el script de inferencia, se deberá cargar el modelo previamente entrenado y utilizarlo para realizar predicciones sobre los datos preprocesados. Al final, se devolverá la predicción siguiendo los criterios de formato establecidos en el contrato.

Por último, en el script de predict_file, se unifica todo lo anterior con la lectura del archivo CSV, la validación de cada fila, la ejecución de las predicciones y la generación de un nuevo CSV con los resultados.

## Entrega

Entrega una captura o exportación con:

- La tabla de contrato.
- Las respuestas de las prácticas A, B y C.
- El orden de las 11 características.
- Un ejemplo de entrada inválida.
- Un contrato de salida.
- Tu plan de implementación para `contracts.py`, `preprocess.py`, `inference.py` y `predict_file.py`.

### Tres criterios de aceptación para el script de la clase 2

1. **Validación de entradas:** El script acepta una fila válida con las 11
   variables esperadas y rechaza cualquier fila con columnas desconocidas, valores ausentes o valores fuera de rango.

2. **Siempre el mismo orden:** El script construye siempre el vector de
   entrada con las 11 variables en el orden exacto de `FEATURE_NAMES`,
   independientemente del orden en el que aparezcan las columnas en el CSV.

3. **Salida con contrato estable:** para cada fila válida, el script genera una
   salida con `sample_id`, `quality_band`, `confidence`,
   `model_version` y `preprocessing_version`. `quality_band` solo puede tomar
   los valores `needs_review`, `acceptable` o `excellent`, y `confidence` debe
   estar entre 0 y 1.